### Notebook for parsing deployment time

In [ ]:
import pandas as pd

from pathlib import Path
from loguru import logger

#### List all table files in directory

In [ ]:
TABLE_DIR: Path = Path("/home/martin/data/acfr_telemetry_tables")
if not TABLE_DIR.is_dir():
    raise ValueError(f"path is not a directory: {TABLE_DIR}")

table_files: list[Path] = [
    path
    for path in TABLE_DIR.iterdir()
    if path.is_file() and path.suffix == ".csv"
]

logger.info(f"Table files: {len(table_files)}")

#### Group tables by prefix

In [ ]:
prefixes: list[str] = [
    "qdch0ftq_20100428_020202",
    "qdch0ftq_20110415_020103",
    "qdch0ftq_20120430_002423",
    "qdch0ftq_20130406_023610",
    "qdchdmy1_20110416_005411",
    "qdchdmy1_20120501_071203",
    "qdchdmy1_20130406_081713",
    "qdchdmy1_20170525_234624",
    "r7jjskxq_20101023_210332",
    "r7jjskxq_20121013_060425",
    "r7jjskxq_20131022_004934",
    "r29mrd5h_20090612_225306",
    "r29mrd5h_20090613_100254",
    "r29mrd5h_20110612_033752",
    "r29mrd5h_20130611_002419",
    "r23685bc_20100605_021022",
    "r23685bc_20120530_233021",
    "r23685bc_20140616_225022",
]

table_file_groups: dict[str, list[Path]] = dict()
for prefix in prefixes:
    matches: list[Path] = [
        path for path in table_files if path.name.startswith(prefix)
    ]
    table_file_groups[prefix] = matches

for prefix, files in table_file_groups.items():
    logger.info(f"Prefix: {prefix}, files: {len(files)}")

#### Read tables and merge

In [ ]:
group_dfs: dict[str, pd.DataFrame] = dict()
for prefix, files in table_file_groups.items():
    # TODO: Read and concatenate pandas data frames
    df: pd.DataFrame = pd.concat([pd.read_csv(path) for path in files])
    group_dfs[prefix] = df

#### TODO: Parse data

In [ ]:
from tqdm.auto import tqdm

rows: list[dict] = list()
for prefix, df in tqdm(group_dfs.items(), desc="Processing deployments..."):
    # Select timestamp column
    selected: pd.DataFrame = df[["timestamp"]].copy()
    selected["datetime"] = pd.to_datetime(selected["timestamp"], unit="s")
    selected["prefix"] = prefix

    min_index: int = selected["timestamp"].idxmin()
    max_index: int = selected["timestamp"].idxmax()

    start_row = selected.iloc[min_index]
    end_row = selected.iloc[max_index]

    min_timestamp: float = selected["timestamp"].min()
    max_timestamp: float = selected["timestamp"].max()

    row: dict = {
        "prefix": prefix,
        "epoch_start": start_row["timestamp"],
        "epoch_end": end_row["timestamp"],
        "datetime_start": start_row["datetime"],
        "datetime_end": end_row["datetime"],
    }

    rows.append(row)


deployment_data: pd.DataFrame = pd.DataFrame(rows)
deployment_data: pd.DataFrame = deployment_data.sort_values(by="prefix")

# Round start and end time to nearest seconds
deployment_data["datetime_start"] = deployment_data["datetime_start"].dt.floor(
    "1s"
)
deployment_data["datetime_end"] = deployment_data["datetime_end"].dt.ceil("1s")

logger.info(deployment_data.head())
logger.info(deployment_data.dtypes)

In [ ]:
export_file: Path = Path(
    "/home/martin/data/acfr_telemetry_tables/deployment_times.csv"
)
deployment_data.to_csv(
    export_file,
    index=False,
)